# HSV Color Picker using OpenCV
**Full Coding available at [HSV_Color_Picker.py](HSV_Color_Picker.py)**
<br>
This is the documentation for HSV Color Picker using OpenCV trackbars for real-time HSV value adjustment.

The HSV Color Picker is a tool for finding the optimal HSV (Hue, Saturation, Value) range for color detection in computer vision applications. This script uses trackbars to interactively adjust HSV values and provides real-time visual feedback for color filtering.

***Libraries Used In The Script***
- OpenCV (cv2) - For image processing and GUI trackbars
- Numpy (np) - For numerical operations and array handling
- Picamera2 - Raspberry Pi camera interface
- libcamera - Camera control library
- time - For time-related operations

## Installing Libraries & Dependencies 
- Built-in Libraries:
    - Picamera2 (Pre-installed on Raspberry Pi OS)
    - libcamera (Pre-installed on Raspberry Pi OS)
    - time (Built-in Python library)
    - Numpy (Usually pre-installed)
- Libraries Requiring Installation:
    - OpenCV

## Installing Required Libraries 
1. **OpenCV** [OpenCV.org](https://docs.opencv.org/4.x/d2/de6/tutorial_py_setup_in_ubuntu.html)
    - In the terminal type: `pip install opencv-python` 
    - To verify installation, in a Python file type: <br>
    `import cv2` <br>
    `print(cv2.__version__)`
    - This will print the version of OpenCV library installed

## Let's Start Coding ! 
### 1. Import all the required libraries 
- OpenCV (cv2) - For image processing and GUI trackbars
- Numpy (np) - For numerical operations and array handling
- Picamera2 - For Raspberry Pi camera control
- libcamera - For camera configuration and controls
- time - For time operations (if needed)

In [ ]:
#finding hsv range of target object(pen)
import cv2
import numpy as np
import time
from picamera2 import Picamera2
from libcamera import controls, Transform

### 2. Define Callback Function and Initialize Camera
- Create a callback function for trackbar events 
    ```python
    # A required callback method that goes into the trackbar function.
    def nothing(x):
        pass
    ```
- Initialize the Picamera2 object <br>
    ```python
    picam2 = Picamera2()
    ```
- Configure camera settings:
    - **Format**: RGB888 for color processing
    - **Size**: 640x480 pixels
    - **Transform**: Vertical flip for correct orientation
    ```python
        config = picam2.create_preview_configuration(main={"format": 'RGB888', "size": (640, 480)},transform=Transform(vflip=1))
        picam2.configure(config)
    ```
- Start camera capture with continuous autofocus
    ```python 
    picam2.start() 
    picam2.set_controls({"AfMode": controls.AfModeEnum.Continuous})
    ```

In [ ]:
# A required callback method that goes into the trackbar function.
def nothing(x):
    pass

# Initializing the webcam feed.
picam2 = Picamera2()
config = picam2.create_preview_configuration(main={"format": 'RGB888', "size": (640, 480)},transform=Transform(vflip=1))
picam2.configure(config)
picam2.start() 
picam2.set_controls({"AfMode": controls.AfModeEnum.Continuous})

### 3. Create Trackbar Window and Controls
Create a GUI window with trackbars for adjusting HSV ranges in real-time:
- **Trackbar Window**: Named "Trackbars" for housing all controls
```python
    # Create a window named trackbars.
    cv2.namedWindow("Trackbars")
```
- **HSV Range Controls**:
    - L-H, L-S, L-V: Lower bounds for Hue (0-179), Saturation (0-255), Value (0-255)
    - U-H, U-S, U-V: Upper bounds for Hue (0-179), Saturation (0-255), Value (0-255) <br>
        ```python
        cv2.createTrackbar("L - H", "Trackbars", 0, 179, nothing)
        cv2.createTrackbar("L - S", "Trackbars", 0, 255, nothing)
        cv2.createTrackbar("L - V", "Trackbars", 0, 255, nothing)
        cv2.createTrackbar("U - H", "Trackbars", 179, 179, nothing)
        cv2.createTrackbar("U - S", "Trackbars", 255, 255, nothing)
        cv2.createTrackbar("U - V", "Trackbars", 255, 255, nothing)
        ```
- **Initial Values**: Set to full range (0 for lower bounds, max for upper bounds)

In [ ]:
# Create a window named trackbars.
cv2.namedWindow("Trackbars")

# Now create 6 trackbars that will control the lower and upper range of 
# H,S and V channels. The Arguments are like this: Name of trackbar, 
# window name, range,callback function. For Hue the range is 0-179 and
# for S,V its 0-255.
cv2.createTrackbar("L - H", "Trackbars", 0, 179, nothing)
cv2.createTrackbar("L - S", "Trackbars", 0, 255, nothing)
cv2.createTrackbar("L - V", "Trackbars", 0, 255, nothing)
cv2.createTrackbar("U - H", "Trackbars", 179, 179, nothing)
cv2.createTrackbar("U - S", "Trackbars", 255, 255, nothing)
cv2.createTrackbar("U - V", "Trackbars", 255, 255, nothing)

### 5. The `While True` Loop 
- All the code will keep looping in here 
    - Get the capture image form the camera<br>
        ```python
        frame = cam.capture_array()
        ```
    - Convert the color format from RGB to HSV <br>
        ```python
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        ```
    - From the trackbars get all the values (Trackbar Postion) <br>
        ```python
        l_h = cv2.getTrackbarPos("L - H", "Trackbars")
        l_s = cv2.getTrackbarPos("L - S", "Trackbars")
        l_v = cv2.getTrackbarPos("L - V", "Trackbars")
        u_h = cv2.getTrackbarPos("U - H", "Trackbars")
        u_s = cv2.getTrackbarPos("U - S", "Trackbars")
        u_v = cv2.getTrackbarPos("U - V", "Trackbars")
        ```
    - Insert the trackbars value to the array<br>
        ```python
        lower_range = np.array([l_h, l_s, l_v])
        upper_range = np.array([u_h, u_s, u_v])
        ```
    - Filter out the color based on the HSV array (range)<br>
        ```python
        mask = cv2.inRange(hsv, lower_range, upper_range)
        ```
    - Invert the filter (So that we can visualise which object is selected based on the color) <br>
        ```python
        res = cv2.bitwise_and(frame, frame, mask=mask)
        ```
    - Convert the filtered image to a3 channel image (Contains R, G, and B) <br>
        ```python
        mask_3 = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
        ```
    - Combine all the result in one window. <br>
        ```python
        stacked = np.hstack((mask_3,frame,res))
        ```
    - Show all the results and trackbars <br>
        ```python
        cv2.imshow('Trackbars',cv2.resize(stacked,None,fx=0.4,fy=0.4))
        ```
    - If the user press the Escape key (Esc) the programe will quit <br>
        ```python
        key = cv2.waitKey(1)
        if key == 27:
            break
        ```
    - Save the adjusted value and display it. <br>
        ```python
        if key == ord('s'):
            thearray = [[l_h,l_s,l_v],[u_h, u_s, u_v]]
            print(thearray)
            # Also save this array as penval.npy
            np.save('hsv_value',thearray)
            break
        ```

In [ ]:
while True:
    
    # Start reading the webcam feed frame by frame.
    frame = picam2.capture_array()
    
    # Convert the BGR image to HSV image.
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    # Get the new values of the trackbar in real time as the user changes 
    # them
    l_h = cv2.getTrackbarPos("L - H", "Trackbars")
    l_s = cv2.getTrackbarPos("L - S", "Trackbars")
    l_v = cv2.getTrackbarPos("L - V", "Trackbars")
    u_h = cv2.getTrackbarPos("U - H", "Trackbars")
    u_s = cv2.getTrackbarPos("U - S", "Trackbars")
    u_v = cv2.getTrackbarPos("U - V", "Trackbars")
 
    # Set the lower and upper HSV range according to the value selected
    # by the trackbar
    lower_range = np.array([l_h, l_s, l_v])
    upper_range = np.array([u_h, u_s, u_v])
    
    # Filter the image and get the binary mask, where white represents 
    # your target color
    mask = cv2.inRange(hsv, lower_range, upper_range)
 
    # You can also visualize the real part of the target color (Optional)
    res = cv2.bitwise_and(frame, frame, mask=mask)
    
    # Converting the binary mask to 3 channel image, this is just so 
    # we can stack it with the others
    mask_3 = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    
    # stack the mask, orginal frame and the filtered result
    stacked = np.hstack((mask_3,frame,res))
    
    # Show this stacked frame at 40% of the size.
    cv2.imshow('Trackbars',cv2.resize(stacked,None,fx=0.4,fy=0.4))
    
    # If the user presses ESC then exit the program
    key = cv2.waitKey(1)
    if key == 27:
        break
    
    # If the user presses `s` then print this array.
    if key == ord('s'):
        
        thearray = [[l_h,l_s,l_v],[u_h, u_s, u_v]]
        print(thearray)
        
        # Also save this array as penval.npy
        np.save('hsv_value',thearray)
        break

### 5. Wrapping Things Up 
When the program exits (user presses ESC or 's'), it's important to properly clean up resources:

- **Stop the Camera**: Release the camera resource to make it available for other applications
    ```py
    picam2.stop()
    ```
    
- **Destroy All Windows**: Close all OpenCV windows created by the program
    ```py
    cv2.destroyAllWindows()
    ```

This ensures that:
- The camera is properly released and can be used by other programs
- All GUI windows (including trackbar window) are closed cleanly
- System resources are freed up properly
- HSV values are saved (if 's' was pressed) for future use in object detection

In [ ]:
# Release the camera & destroy the windows.
picam2.stop()
cv2.destroyAllWindows()